### 1.3.7.8. Reverse-Mode Automatic Differentiation

$$
\bar{v} = \frac{\partial L}{\partial v}: \qquad
\bar{v}_{\text{parent}} \mathrel{+}= \bar{v}_{\text{child}}\,\frac{\partial\,v_{\text{child}}}{\partial\,v_{\text{parent}}}
\quad (\text{backward sweep, seed } \bar{L} = 1) .
$$

**Explanation:**

Reverse-mode AD — backpropagation — computes the gradient of a scalar output by a forward pass that records the computation (the Wengert list) followed by a backward pass that accumulates *adjoints* $\bar v = \partial L/\partial v$ from output to inputs, applying the [chain rule](../05_Multivariable_Differential_Calculus/08_multivariable_chain_rule.ipynb) at each node. Seeding $\bar L = 1$ and sweeping backward yields the full [gradient](../05_Multivariable_Differential_Calculus/03_gradient.ipynb) in a single pass, independent of the number of inputs. This is the [VJP](./06_jacobian_vector_and_vector_jacobian_products.ipynb) primitive and the reason training models with millions of parameters is feasible — one backward pass returns every partial derivative.

**Properties:**
- Cost of the gradient is a small constant multiple of one function evaluation, for any number of inputs.
- Memory grows with the computation graph, since intermediate values are stored for the backward sweep.

**Numerical Example:**

Compute $\nabla f$ for $f(x, y) = xy + \sin x$ at $(x, y) = (1, 2)$. Forward pass records intermediates; the backward pass seeds $\bar f = 1$ and propagates:

$$
v_3 = xy,\quad v_4 = \sin x,\quad f = v_3 + v_4 .
$$

$$
\bar v_3 = 1,\ \bar v_4 = 1; \quad
\bar x = \bar v_3\cdot y + \bar v_4\cdot\cos x = y + \cos x,\quad
\bar y = \bar v_3\cdot x = x .
$$

At $(1, 2)$: $\partial f/\partial x = 2 + \cos 1 \approx 2.540$ and $\partial f/\partial y = 1$, matching $\nabla f = [y + \cos x,\ x]^\top$.

In [1]:
import math

x_value, y_value = 1.0, 2.0

product_term = x_value * y_value
sine_term = math.sin(x_value)
output = product_term + sine_term

output_adjoint = 1.0
product_adjoint = output_adjoint * 1.0
sine_adjoint = output_adjoint * 1.0
x_adjoint = product_adjoint * y_value + sine_adjoint * math.cos(x_value)
y_adjoint = product_adjoint * x_value

print("forward: f(1,2)        =", output)
print("backward ∂f/∂x         =", x_adjoint)
print("backward ∂f/∂y         =", y_adjoint)
print("check [y+cos x, x] at (1,2) =", y_value + math.cos(x_value), x_value)

forward: f(1,2)        = 2.8414709848078967
backward ∂f/∂x         = 2.5403023058681398
backward ∂f/∂y         = 1.0
check [y+cos x, x] at (1,2) = 2.5403023058681398 1.0


**Equivalent CasADi implementation:**

`ca.gradient` runs reverse-mode AD over the graph and returns the same gradient in one backward sweep.

In [2]:
import casadi as ca

variables = ca.SX.sym("x", 2)
f = variables[0] * variables[1] + ca.sin(variables[0])
gradient_function = ca.Function("grad", [variables], [ca.gradient(f, variables)])

print("∇f(1,2) via reverse-mode AD =", gradient_function([1, 2]))

∇f(1,2) via reverse-mode AD = [2.5403, 1]


**References:**

[📘 Griewank, A., & Walther, A. (2008). *Evaluating Derivatives* (2nd ed.). SIAM.](https://epubs.siam.org/doi/book/10.1137/1.9780898717761)  
[📗 Baydin, A. G., Pearlmutter, B. A., Radul, A. A., & Siskind, J. M. (2018). *Automatic Differentiation in Machine Learning: a Survey*. JMLR 18(153).](https://jmlr.org/papers/v18/17-468.html)

---

[⬅️ Previous: Forward-Mode Automatic Differentiation](./07_forward_mode_automatic_differentiation.ipynb) | [Next: Numerical, Symbolic, and Automatic Differentiation ➡️](./09_numerical_symbolic_and_automatic_differentiation.ipynb)